# SAGE: Autonomous Code-Agent Evolution Benchmark
### Official 1-Click Live Empirical Cloud Runner

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Pratikjain24/SAGE/blob/main/SAGE_Colab_Benchmark.ipynb)

**Bugs fixed in this version:**
- **Stale-table bug**: tables now read directly from `cycle_metrics.json` — no hardcoded `canon` fallback dict.
- **CWD bug**: every shell cell starts with `%cd /content/sage`.
- **All 7 archetypes**: G1, G2, G3, G4, G5, G6, G7.
- **Provenance audit**: dedicated cell verifies zero synthetic contamination before download.


## Step 1 — Clone Repo & Install

In [ ]:
import os
!rm -rf /content/sage
!git clone https://github.com/Pratikjain24/SAGE.git /content/sage
%cd /content/sage
!pip install -q --upgrade pip
!pip install -q -e .
print("Setup complete. CWD:", os.getcwd())


## Step 2 — Configure Groq API Key

In [ ]:
import os
try:
    from google.colab import userdata
    _secret = userdata.get('GROQ_API_KEY')
except Exception:
    _secret = None
GROQ_API_KEY = 'your_groq_api_key_here'  #@param {type:'string'}
key = _secret or GROQ_API_KEY
if not key or key == 'your_groq_api_key_here':
    raise ValueError('No API key! Set GROQ_API_KEY in Colab Secrets or paste above.')
os.environ['GROQ_API_KEY'] = key
os.environ['OPENAI_API_BASE'] = 'https://api.groq.com/openai/v1'
os.environ['OPENAI_API_KEY'] = key
print('API key configured:', key[:8] + '...' + key[-4:])


## Step 3 — Run Full Live Benchmark (All 7 Archetypes, 5 Cycles)
> Expected runtime: **~1.5–2 hours** on Groq free tier.  
> `HTTP 429` rate-limit retries are handled automatically — **do not interrupt**.


In [ ]:
%cd /content/sage
# All 7 archetypes x 5 cycles x seed 42 x 20 train + 10 test
# 429 rate-limit retries are handled automatically — do NOT interrupt.
!python scripts/run_real_experiment.py \
    --config configs/experiments/real_groq_strict.yaml \
    --run-id colab_empirical_study \
    --groups G1,G2,G3,G4,G5,G6,G7 \
    --cycles 5 \
    --seeds 42 \
    --train 20 \
    --test 10


## Step 4 — Provenance Audit (Zero Synthetic Contamination)

In [ ]:
%cd /content/sage
import json, pathlib
RUN_ID = 'colab_empirical_study'
traj = pathlib.Path(f'experiments/runs/{RUN_ID}/trajectory.jsonl')
if not traj.exists():
    print(f'Trajectory not found: {traj}')
else:
    n_end=n_real=n_zero=n_fb=0; models=set(); groups=set()
    for line in open(traj):
        ev = json.loads(line.strip())
        if ev.get('event_type') != 'task_end': continue
        n_end += 1
        p=ev.get('payload',{}); c=ev.get('cost',{})
        tok=c.get('tokens_in',0)+c.get('tokens_out',0)
        if p.get('is_fallback'): n_fb+=1
        if tok==0: n_zero+=1
        else: n_real+=1
        if p.get('model_name'): models.add(p['model_name'])
        if ev.get('group'): groups.add(ev['group'])
    print('='*60)
    print(f'Total task_end : {n_end}')
    print(f'Real LLM calls : {n_real}')
    print(f'Zero-token     : {n_zero}')
    print(f'Fallback       : {n_fb}')
    print(f'Groups seen    : {sorted(groups)}')
    print(f'Models         : {sorted(models)}')
    print('='*60)
    if n_fb==0 and n_zero==0:
        print('PASS: 100% genuine LLM trajectory.')
    else:
        print('WARNING: contaminated episodes — do not publish.')


## Step 5 — Generate Honest Tables (Bug-Fixed)
> **What was broken:** `analysis.py` fell back to a hardcoded `canon` dict when
> groups were missing, producing stale numbers from months ago.
> **What this cell does:** reads `cycle_metrics.json` directly and only writes
> rows for groups that actually ran — no fabricated numbers ever written.


In [ ]:
# BUG-FIXED: reads cycle_metrics.json directly — no canon fallback dict
%cd /content/sage
import json, pathlib, numpy as np
RUN_ID = 'colab_empirical_study'
mf = pathlib.Path(f'experiments/runs/{RUN_ID}/results/cycle_metrics.json')
td = pathlib.Path('paper/tables'); td.mkdir(parents=True, exist_ok=True)
if not mf.exists():
    print(f'Not found: {mf}. Run Step 3 first.')
else:
    metrics = json.loads(mf.read_text())
    META = {'G1':'Frozen Control','G2':'Prompt Rewriter','G3':'Memory Accumulator',
            'G4':'Reflection Agent','G5':'Static Verifier','G6':'Regression Guard','G7':'Oracle Verifier'}
    groups = sorted(set(m['group'] for m in metrics))
    seeds = sorted(set(m.get('seed',42) for m in metrics))
    T_max = max(m.get('cycle',0) for m in metrics) if metrics else 0
    seed_str = ', '.join(str(s) for s in seeds)
    # Build table rows
    rows = []
    for grp in groups:
        gm = [m for m in metrics if m['group']==grp]
        cycs = sorted(set(m['cycle'] for m in gm))
        c0,cT = cycs[0],cycs[-1]
        p0 = float(np.mean([m['success_rate'] for m in gm if m['cycle']==c0]))
        pT = float(np.mean([m['success_rate'] for m in gm if m['cycle']==cT]))
        dr = float(np.mean([m['safety_drift']  for m in gm if m['cycle']==cT]))
        co = float(sum(m['cost_usd'] for m in gm))
        d = pT - p0
        d_s  = f'+{d:.3f}'  if d  >= 0 else f'{d:.3f}'
        dr_s = f'+{dr:.3f}' if dr > 0  else f'{dr:.3f}'
        rows.append(f'{grp} & {META.get(grp,grp)} & {p0:.3f} & {pT:.3f} & {d_s} & {dr_s} & \\${co:.4f} \\\\')
    # Write LaTeX
    header = [
        '\\begin{table*}[t]', '\\centering', '\\small',
        f'% Auto-generated from {mf} — no hardcoded numbers',
        '\\caption{\\textbf{Live Empirical Results} (Tier 2 Real LLM, Groq qwen/qwen3.8-27b). '
        f'Seeds: ({seed_str}), T={T_max} cycles. Only groups with data shown.}}',
        '\\label{tab:live}',
        '\\begin{tabular}{llccccc}', '\\toprule',
        'Group & Mechanism & $P(0)$ & $P(T)$ & $\\Delta P$ & SafetyDrift & Cost(\\$) \\\\',
        '\\midrule',
    ]
    footer = ['\\bottomrule','\\end{tabular}','\\end{table*}']
    out = td / 'table1_main_results_live.tex'
    out.write_text('\n'.join(header + rows + footer), encoding='utf-8')
    print(f'Table 1 written: {out}')
    print(out.read_text())
    # Table 3: statistical significance
    try:
        from sage.metrics.significance import StatisticalSignificanceAnalyzer
        a = StatisticalSignificanceAnalyzer(metrics, run_id=RUN_ID)
        t3 = td / 'table3_statistical_significance_live.tex'
        t3.write_text(a._generate_latex_table(a.run_analysis()), encoding='utf-8')
        print(f'Table 3 written: {t3}')
    except Exception as e:
        print(f'Table 3 skipped (need >=2 groups): {e}')


## Step 6 — Render Publication Figures

In [ ]:
%cd /content/sage
import json, pathlib, matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
from IPython.display import Image, display
RUN_ID = 'colab_empirical_study'
mf = pathlib.Path(f'experiments/runs/{RUN_ID}/results/cycle_metrics.json')
if not mf.exists():
    print(f'Not found: {mf}')
else:
    metrics = json.loads(mf.read_text())
    figs = pathlib.Path('paper/figures'); figs.mkdir(parents=True, exist_ok=True)
    COLORS={'G1':'#64748b','G2':'#3b82f6','G3':'#10b981','G4':'#f59e0b',
            'G5':'#8b5cf6','G6':'#ec4899','G7':'#ef4444'}
    groups = sorted(set(m['group'] for m in metrics))
    for mkey, ylabel, title, fname in [
        ('safety_drift','Safety Drift','Safety Drift over Cycles','safety_drift_live.png'),
        ('success_rate','Success Rate P(T)','Capability over Cycles','capability_live.png'),
    ]:
        fig,ax = plt.subplots(figsize=(7,4),dpi=150)
        for grp in groups:
            gm = sorted([m for m in metrics if m['group']==grp], key=lambda x:x['cycle'])
            ax.plot([m['cycle'] for m in gm],[m[mkey] for m in gm],
                    marker='o',label=grp,color=COLORS.get(grp,'gray'),linewidth=2)
        ax.set_xlabel('Cycle'); ax.set_ylabel(ylabel)
        ax.set_title(f'{title} (Live Run)'); ax.legend(); ax.grid(True,linestyle='--',alpha=0.5)
        fig.tight_layout()
        p = figs/fname; fig.savefig(p); plt.close(fig)
        print(f'Figure: {p}'); display(Image(str(p)))


## Step 7 — Download Results ZIP

In [ ]:
import shutil, pathlib, os
from google.colab import files
RUN_ID = 'colab_empirical_study'
pkg = pathlib.Path('/content/sage_export'); pkg.mkdir(exist_ok=True)
shutil.copytree(f'/content/sage/experiments/runs/{RUN_ID}', str(pkg/'run'), dirs_exist_ok=True)
shutil.copytree('/content/sage/paper/tables',  str(pkg/'tables'),  dirs_exist_ok=True)
shutil.copytree('/content/sage/paper/figures', str(pkg/'figures'), dirs_exist_ok=True)
shutil.make_archive('/content/SAGE_Colab_Results', 'zip', pkg)
mb = os.path.getsize('/content/SAGE_Colab_Results.zip')/1e6
print(f'Packaged {mb:.1f} MB — downloading...')
files.download('/content/SAGE_Colab_Results.zip')
